# Download the dataset


# Confident Legal Research Application

- Main stakeholder/consumer: Junior Legal Assistant
- Distributed: No


**Goal:Improve quality & speed of case research for junior assistants**

## Baseline System

To create an evaluation process we would need to have some baseline system in place. That is why we are creating a **Naive implementation** with the following initial setup:

- One dense vector embedding (`BAAI/bge-small-en-v1.5`, 384-dim)
- Search function with no reranker
- Naive RAG pipeline

In [1]:
%load_ext autoreload
%autoreload 2

### Initialize dataset

We will use the famous CLERC dataset for this tutorial. Two reasons:
1. There exist queries in the dataset that we can use as a baseline for complex law queries 
2. The combination of queries/positive/negative responses provides a good baseline for the *golden set* that could be used for testing

In [2]:
from src.dataset import load_clerc


clerc_data = load_clerc("train", limit=2000)

clerc_data

Loading dataset shards:   0%|          | 0/31 [00:00<?, ?it/s]

Dataset to pydantic transformation:   0%|          | 0/2000 [00:00<?, ?it/s]

ClercSplit(corpus=41167 docs, queries=2000, qrels=2000)

In [3]:
sample_tuple = clerc_data.sample()

sample_tuple

(Query(query_id='766072', query="wells, other natural deposits, and timber, there shall be allowed as a deduction in computing taxable income a reasonable allowance for depletion and for depreciation of improvements, according to the peculiar conditions in each case; such reasonable allowance in all cases to be made under regulations prescribed by the Secretary or his delegate.” As stated in Parsons v. Smith, 359 U.S. 215, 220, 79 S.Ct. 656, 660, 3 L.Ed.2d 747 (1959): “The purpose of the deduction for depletion is plain and has been many times declared by this Court. ‘[It] is permitted in recognition of the fact that the mineral deposits are wasting assets and is intended as compensation to the owner for the part used up in production.’  REDACTED 366, 58 S.Ct. 616, 618, 82 L.Ed. 897. And see United States v. Ludey, 274 U.S. 295, 302, 47 S.Ct. 608, 610, 71 L.Ed. 1054; Helvering v. Elbe Oil Land Development Co., 303 U.S. 372, 375, 58 S.Ct. 621, 622, 82 L.Ed. 904; Anderson v. Helvering, 3

### Initialize vectors 

- use `qdrant_client` for storing vectors
- use `fastembed` for the simplest embeddings generation pipelines


We will also store the embeddings locally and try to handle them in parallel.

>NB!: This notebook is optimized to run on macbook devices. Try to adapt embedding config to your specific hardware

In [4]:
from qdrant_client import QdrantClient
import os

qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')

client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)

In [5]:
# constants
COLLECTION_NAME = 'lawyer_citations'
DENSE_MODEL = "BAAI/bge-small-en-v1.5"
SIZE = 384

In [6]:
from src.indexing import EmbeddingConfig
from qdrant_client.models import Distance


config = EmbeddingConfig(
    name='dense_base',
    model_id=DENSE_MODEL,
    kind='dense',
    size=SIZE,
    distance=Distance.COSINE,
    providers=["CoreMLExecutionProvider", "CPUExecutionProvider"],
    parallel=None,
)

In [7]:
from src.indexing import EmbeddingCache
embedding_cache = EmbeddingCache()

In [9]:
from src.indexing import DocumentIndexer

document_vectors = DocumentIndexer(
    client,
    COLLECTION_NAME,
    embeddings=[config],
    cache=embedding_cache
)

In [10]:
document_vectors.ensure_collection()

In [20]:
document_vectors.upload(items=list(clerc_data.corpus.values()), batch_size=128)

embed:dense_base:   0%|          | 0/41167 [00:00<?, ?it/s]

In [11]:
client.get_collection(COLLECTION_NAME).points_count, (client.get_collection(COLLECTION_NAME).points_count or 0) > 0

(41167, True)

### Create simple semantic search and RAG pipelines

- `DenseSearcher` is the simplest implementation of the semantic search: no sparse vectors, multivectors, no reranking. Just query API
- `LegalAssistant` - uses `DenseSearcher` for the retrieval and generates the results based on that. We use `anthropic api` with `lite-llm`

In [12]:
from src.search import DenseSearcher
from src.rag import LegalAssistant

qa = clerc_data.sample()
searcher = DenseSearcher(client, COLLECTION_NAME, config)
legal_assistant = LegalAssistant(searcher)


In [13]:
result = searcher.search(qa[0].query)
result[0].text, qa[1].text

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

('supra, at 430 and 434 (Winter, J., concurring and dissenting). 3. To the extent the complaint seeks equitable relief under § 1983 against the members of the Board in their official capacities it may be prosecuted, for such municipal officers are “persons” within the meaning of § 1983. Harper v. Kloster, 486 F.2d 1134, 1138 (4th Cir. 1973); Incarcerated Men of Allen Co. Jail v. Fair, 507 F.2d 281, 287-88 (6th Cir. 1974); Ybarra v. City of Town of Los Altos Hills, 503 F.2d 250, 253 (9th Cir. 1974). For cases in which equitable relief under § 1983 was allowed against defendants in their capacity as municipal officials, see, e. g., Griffin v. County School Board, 377 U.S. 218, 84 S.Ct. 1226, 12 L.Ed.2d 256 (1964); Douglas v. City of Jeannette, 319 U.S. 157, 63 S.Ct. 877, 87 L.Ed. 1324 (1943), cited in Monroe v. Pape, 365 U.S. 167, 191, n. 50, 81 S.Ct. 473. But see Moye v. City of Raleigh, 503 F.2d 631, 635, n. 11 (4th Cir. 1974) (caveat); Bennett v. Gravelle, 323 F.Supp. 203, 211 (D.Md.1

In [14]:
answer = legal_assistant.ask("My clients landlord wants him to pay for the wall that was broken. No hand-in protocol was signed. Give me cases with similar problems")

print(answer)

Based solely on the case excerpts provided, none of the retrieved cases directly address the situation you have described — namely, a landlord seeking payment for a damaged wall where no hand-in (move-out inspection) protocol was signed. The excerpts cover the following unrelated legal topics:

1. Doc 9457344: A bankruptcy/debt dispute involving furniture repossession and post-petition claims — not related to property damage or move-out protocols.
2. Doc 14718586: A case about agricultural diversion payments and a tenant's obligation to assign payments to a landlord — not related to property damage or inspection protocols.
3. Doc 8705202: A government contract dispute involving equitable adjustments and administrative finality — not related to landlord-tenant property damage.
4. Doc 3706151: A landlord-tenant case under the Emergency Price Control Act of 1942 concerning rent overcharges — not related to property damage or move-out inspections.
5. Doc 4359176: A bankruptcy/receivership 

#### Simple baseline - LGTM dataset 

We can always start with the simplest approach, to understand if we even need to delve deep into the evaluations - a good old **Looks Good to Me (LGTM later)**. Stated simply: we can use either LLM or human to define if the queries are actually making good or no. This is a good starting point in case when decision whether system is working according to requirements or not.

In [15]:
lgtm_dataset = [
    'The plaintiff seeks expectation damages after the defendant failed to deliver goods under a commercial supply agreement. What precedents discuss breach of contract, expectation damages, and the duty to mitigate losses?', # Contract Law
    'A customer was injured on business premises after multiple intervening events contributed to the accident. What cases analyze duty of care, proximate cause, and foreseeability?', # Tort Law
    "My clients landlord wants him to pay for the wall that was broken. No hand-in protocol was signed. Give me cases with similar problems", # Tenant Law
]

#### Contract Law

In [16]:
answer = legal_assistant.ask(lgtm_dataset[0])

print(answer)

The excerpts address several relevant legal principles concerning breach of contract, expectation damages, and the duty to mitigate. Here is a summary of the key precedents and rules:

1. Proof of Damages and Reasonable Certainty

Under New York law, once it is established that a breach of contract has caused damages, the plaintiff need only prove the amount of those damages with "reasonable certainty" — not absolute precision. The foundational rule, articulated in Wakeman v. Wheeler & Wilson Mfg. Co., 101 N.Y. 205, 209 (1886), is that "[w]hen it is certain that damages have been caused by a breach of contract, and the only uncertainty is as to their amount, there can rarely be good reason for refusing, on account of such uncertainty, any damages whatever for the breach." Courts have further held that "[a] person violating his contract should not be permitted entirely to escape liability because the amount of the damages which he has caused is uncertain," and that "the wrongdoer must s

### Tort Law 


In [17]:
answer = legal_assistant.ask(lgtm_dataset[1])

print(answer)

Based on the provided excerpts, the following analysis applies to duty of care, proximate cause, and foreseeability in the context of customer injuries on business premises:

1. Duty of Care: Tort liability fundamentally depends on the violation of a duty of care owed to the injured person. The scope of that duty — whether it extends to bystanders, customers, or other classes of plaintiffs — is ordinarily defined by the common law. Legislatures can create new duties of care, but the mere fact that a statute defines due care does not, by itself, create a tort-enforceable duty. Courts have consistently held that a plaintiff must first establish that the defendant owed them a duty before any further negligence analysis can proceed.

2. Proximate Cause: Causation is described as "the most susceptible to summary determination" among all elements necessary to support recovery in a tort action. To demonstrate proximate cause, a plaintiff must show that the defendant, through affirmative actio

In [18]:
answer = legal_assistant.ask(lgtm_dataset[2])

print(answer)

Based solely on the case excerpts provided, none of the retrieved cases directly address the specific legal issue of a landlord seeking payment for property damage (a broken wall) in the absence of a signed hand-in (move-out/check-out) protocol. The excerpts cover unrelated legal matters, including bankruptcy and furniture repossession (doc_id: 9457344), agricultural payment assignments and landlord rent notes (doc_id: 14718586), government contract equitable adjustments (doc_id: 8705202), emergency rent control regulations and overcharges (doc_id: 3706151), and bankruptcy bar orders and asset distribution (doc_id: 4359176). None of these cases involve a landlord's claim for property damage to a rental unit, nor do they address the evidentiary or contractual significance of a missing hand-in or move-out protocol.

Therefore, the provided excerpts do not contain sufficient information to answer your question about cases involving a landlord's claim for wall damage in the absence of a si

#### Results interpetation 

From the first side the AI generated queries are being passed. However, if you ask stronger LLM model to analyze the actual result it will identify two issues with the results if the `lgtm_dataset[0]` and `lgtm_dataset[1]` queries replies:

**Confidence exceeds evidence** (prompt discipline is broken - section 3 and summary are full of speculation in first query) - grave mistake cause it will confuse the Junior Assistants in the same vein it does us. Law needs not just confidence, but a lot of precision

That is why I urge the practitioners to write some queries without using AI if they want to judge manually (AI generated queries -> stronger AI judgements, human generated queries -> both human and AI judgements can work). For the `lgtm_dataset[2]` we see a clear outlier: `Based on the case excerpts provided, none of them directly address the specific legal issue `. This either means that our query is wrong for this dataset (mostly this question was from personal experience in EU not US), however this does not reflect the result => our naive implementation of search still gave wrong results. This is a clear signal now that evaluation is needed to setup a **baseline/AS IS**

### Basis of the evaluation framework

Even on a single query, the retrieval layer is visibly underperforming — by design. The PoC intentionally omits sparse vectors, reranking, and query rewriting so the failure modes those techniques usually hide surface up front.

What saved this run is the LLM: `claude-sonnet-4.6` is strong enough to notice the retrieved excerpts don't answer the question and refuse to fabricate. That safety net is fragile. With a weaker model, an opaque pipeline, or domain experts who can't audit the citation chain, the same retrieval bug becomes invisible — and the system confidently misleads a junior associate who can't yet tell the difference.

## Goal oriented evaluation specification

The evaluation is designed **top-down**: a system that doesn't serve its business goal isn't worth measuring at the component level.

### Inputs/Assumptions

The preconditions feeding into the methodology:
- **GOAL:** Confident Legal Research
- **CONSUMER:** Junior associate — limited domain depth, needs guidance and explanations
- **DISTRIBUTED:** No — single goal, single consumer

In [33]:
from IPython.display import HTML

HTML("""
<div style="text-align: center;">
    <img src="diagrams/eval-inputs.svg" width="300">
</div>
""")

### Goal Decomposition

**Goal decomposition** bridges the abstract business goal and the concrete metrics that measure it. Before we can quantify, we have to be specific about *what the system is* and *what it must do*.

Two questions to anchor the decomposition:
- **Is this a *business* goal?** Yes — it describes the system's *utility*, not its internals.
- **Can we make *component-level* assumptions?** Yes — the system has two: **searcher** (retrieval, with well-understood mechanics) and **rag** (LLM-driven generation).

Refined business objective:

`Improve quality & speed of case research for junior associates`

Technical commitments already baked into the codebase:

`Confident & fast legal agentic search` *(too vague to test as-is — decomposition follows)*

Decomposition is one of the oldest design principles in software engineering. Here we use it to turn the vague tech goal into the evaluation context:

![System Goal Decomposition](diagrams/system-goal-decomposition.svg)

At the system level, three NFRs apply to every sub-system:
- **Trustworthiness** — can the consumer rely on the output without verification, and fail loudly when it can't?
- **Correctness** — does the output match ground truth for the task it owns?
- **Performance** — is it fast enough to be useful in the consumer's loop?

These stay vague until we map them to individual components. Each sub-system has a different consumer, and the consumer determines what each NFR *operationally means*:

![Component Level Goal Decomposition](diagrams/component-level-goal-decomposition.svg)

**Search** is consumed by RAG (programmatic). Its failure surface is narrow — a bad ranked list, an over-confident empty result, or a slow query. Three flat axes are enough.

**RAG** is consumed by a junior associate (human). Its failure surface is richer — confidently-wrong differs sharply from correctly-refused, and citation grounding, hedging, and clarity are all separately damaging when they fail. So the decomposition goes deeper on the RAG side.

**The depth asymmetry is itself the finding:** it tells the reader where the evaluation effort has to concentrate.

One consequence worth calling out early: dense semantic search has no natural zero — every query embeds *somewhere*, every doc embeds *somewhere*, cosine similarity always produces an ordering. So *no relevant doc in the corpus → empty top-K* isn't emergent behavior; it has to be engineered via a calibrated threshold. That's why **Calibration** sits as the headline trustworthiness bullet on the search side.

Without per-component decomposition, a wrong final answer can't be localized to bad retrieval vs. bad synthesis — and what can't be localized can't be fixed. That's exactly what the current PoC shows: RAG had to reject its own retriever's ranking because there was no earlier signal — no calibrated empty, no per-component metric — that would have told the operator the search side was the problem.

### Ranking Quality as example for Non-functional requirements mapped to the metrics


Next part of the tutorial we will focus only on one non-functional requirement that obviously needs some improvement by the simple implementation we have. I am talking about **Ranking quality**, a search component NFR (non-functional requirement, will be used from now on).

What we do is walk through a series of decision questions that help identify the most appropriate metric to use in this situation.

![Evaluation Metric Definition](./diagrams/evaluation-path-to-metric.svg)


The end result is that currently have a clear path why we would measure `NDCG@10` - because:
- We have the golden dataset from `CLERC`
- We want to measure what is the current **Ranking Quality** of the system

We simplified two routes here though: 
- There is no validation of the coverage / leakage / drift for the dataset, that is a separate topic within itself
- We assume that for now it will be enough to ignore the consumer being an LLM instead of a person. Technically this can be a mistake — however, acknowledging that our metric is flawed and doing the best we can with it is better than not measuring at all.

Let us run `NDCG@10` against our dataset and interpret the results

In [19]:
from src.evaluation import evaluate_retrieval

report = evaluate_retrieval(searcher, clerc_data)  # smoke run
print(report)

search:   0%|          | 0/2000 [00:00<?, ?it/s]

Retrieval evaluation (2000 queries, top-10):
  ndcg@10      0.1384
  recall@10    0.2890
  mrr@10       0.0931


Our `ndcg@10` on a 2000-query slice of CLERC **train** is `0.1384`. This lands in the same neighborhood as the published LegalBERT-DPR result of `0.147` — see other [benchmarks](https://www.emergentmind.com/topics/clerc-dataset).

**Caveat:** the published number is measured on the full CLERC *test* split; we're on a subset of *train*. Treat this as a directional sanity check ("we're in the right ballpark for an off-the-shelf dense encoder on legal text"), not a benchmark reproduction. To make the comparison honest, we'd need to rerun on the full test split.

Two reasons the number is low in absolute terms:
1. The legal dataset is domain specific, highly lexical and has complex queries
2. The dataset we are using is actually a trimmed version of the real one. We might expect lower results on the full corpora

### Improvement strategies


We can now use qdrant best practices and ask our LLM to use claude skills to improve the system. Its as easy as prompting into the claude-code the following:
```
Use qdrant skills: https://skills.qdrant.tech/search?query=about%20search%20quality%20improvements to find how to improve the current naive search from @src/search.py
```

![Claude Code Running](./diagrams/improvement-claude-skills.png)




Most probably it will go into one of the following strategies:
- [Use Hybrid Search with BM25](https://qdrant.tech/documentation/search/hybrid-queries/?selector=aHRtbCA%2BIGJvZHkgPiBtYWluID4gc2VjdGlvbiA%2BIGRpdiA%2BIGRpdiA%2BIGRpdjpudGgtb2YtdHlwZSgyKSA%2BIGRpdiA%2BIGRpdjpudGgtb2YtdHlwZSgxKSA%2BIGFydGljbGUgPiBoMjpudGgtb2YtdHlwZSgxKQ%3D%3D&q=Hybrid+Search)
- [Use richer embeddings](https://qdrant.tech/documentation/manage-data/points/)


This time it generated for me Hybrid Search class. Since we plan to also create a new collection we can increase the dense embeddings size as well  

In [8]:
import pandas as pd 
from fastembed import TextEmbedding

supported_models = (
    pd.DataFrame(TextEmbedding.list_supported_models())
    .sort_values("size_in_GB")
    .drop(columns=["sources", "model_file", "additional_files"])
    .reset_index(drop=True)
)
supported_models

,model,description,license,size_in_GB,dim,tasks
0,BAAI/bge-small-en-v1.5,"Text embeddings, Unimodal (text), English, 512...",mit,0.067,384,{}
1,sentence-transformers/all-MiniLM-L6-v2,"Text embeddings, Unimodal (text), English, 256...",apache-2.0,0.090,384,{}
2,BAAI/bge-small-zh-v1.5,"Text embeddings, Unimodal (text), Chinese, 512...",mit,0.090,512,{}
3,snowflake/snowflake-arctic-embed-xs,"Text embeddings, Unimodal (text), English, 512...",apache-2.0,0.090,384,{}
4,jinaai/jina-embeddings-v2-small-en,"Text embeddings, Unimodal (text), English, 819...",apache-2.0,0.120,512,{}
5,BAAI/bge-small-en,"Text embeddings, Unimodal (text), English, 512...",mit,0.130,384,{}
6,nomic-ai/nomic-embed-text-v1.5-Q,"Text embeddings, Multimodal (text, image), Eng...",apache-2.0,0.130,768,{}
7,snowflake/snowflake-arctic-embed-s,"Text embeddings, Unimodal (text), English, 512...",apache-2.0,0.130,384,{}
8,BAAI/bge-base-en-v1.5,"Text embeddings, Unimodal (text), English, 512...",mit,0.210,768,{}
9,sentence-transformers/paraphrase-multilingual-...,"Text embeddings, Unimodal (text), Multilingual...",apache-2.0,0.220,384,{}


In [9]:

DENSE_MODEL_V2  = 'BAAI/bge-base-en-v1.5' #  0.320 GB,	768 DIM	
DENSE_SIZE_V2 = 768
COLLECTION_NAME_V2 = 'trec_dl_v2'
SPARSE_MODEL_V2 = 'Qdrant/bm25' 

In [10]:
from src.indexing import EmbeddingConfig
from qdrant_client.models import Distance
from qdrant_client.models import Modifier


configs_v2 = [
    EmbeddingConfig(
        name='dense_base',
        model_id=DENSE_MODEL_V2,
        kind='dense',
        backend='sentence-transformers',
        size=DENSE_SIZE_V2,
        distance=Distance.COSINE,
    ),
    EmbeddingConfig(
        name="bm25",
        model_id=SPARSE_MODEL_V2,
        kind="sparse",
        modifier=Modifier.IDF
    )
]

In [11]:
from src.indexing import DocumentIndexer

document_indexer_v2 = DocumentIndexer(
    client,
    COLLECTION_NAME_V2,
    embeddings=configs_v2,
    cache=embedding_cache
)

In [9]:
document_indexer_v2.ensure_collection()

In [19]:
document_indexer_v2.upload(items=list(clerc_data.corpus.values()), batch_size=256)

In [12]:
from src.search import HybridSearcher

searcher_v2 = HybridSearcher(
    client, 
    COLLECTION_NAME_V2, 
    configs_v2,
)

In [13]:
qa_v2 = clerc_data.sample()

qa_v2

(Query(query_id='798458', query='the matter be REMANDED to the Secretary for a calculation of benefits and an award thereof to the plaintiff. . The magistrate’s report is attached as an appendix to this memorandum. . I note that there is some question as to the qualification of Dr. Anthony Janelli, the treating physician of Mrs. Scarlata. Other than the testimony of plaintiff herself, there is no evidence in the record which even indicates whether Dr. Janelli is a licensed medical doctor or osteopath. This uncertainty should be clarified on remand. . It should be noted that the opinion of a treating physician is entitled to more weight than that of a one-time examiner. Arnold v. Secretary of HEW, 567 F.2d 258 (4 Cir. 1977);  REDACTED Minor v. Califano [609 F.2d 1265] (E.D.Pa. January, 1979) (Newcomer, J.) (unpublished opinion). Likewise the opinion of the treating physician is entitled to “great weight.” Twardesky v. Weinberger, 408 F.Supp. 842 (W.D.Pa.1976). . Dr. Cander, report of Ju

In [15]:
result = searcher_v2.search(qa_v2[0].query)
result[0].text, qa_v2[1].text

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

('with the FBI agent in charge of the investigation that no electronic surveillance had occurred. Compare In re Vigil, 524 F.2d 209, 214 (10th Cir. 1975), holding that under similar facts not even this was required. At the October 31 contempt hearing, Millow did submit an affidavit to bolster his assertion of illegal wiretapping. That affidavit, however, merely recounted the patently frivolous contention we have dealt with above, namely, that the government had conceded misconduct in its October 29 statement concerning electronic and physical surveillance of Mil-low. Unsupported suspicion and patently frivolous assertions of government misconduct do not constitute a “claim” under § 3504 sufficient to trigger the government’s obligation to disrupt grand jury proceedings and check thoroughly the applicable agency records. We know of no authority to the contrary. While in United States v. Toscanino, 500 F.2d 267, 281 (2d Cir. 1974), we cited with approval language from In re Evans, 146 U.

In [16]:
from src.evaluation import evaluate_retrieval

report_v2 = evaluate_retrieval(searcher_v2, clerc_data)  # smoke run
print(report_v2)

It is top 10 evaluation


search:   0%|          | 0/2000 [00:00<?, ?it/s]

Retrieval evaluation (2000 queries, top-10):
  ndcg@10      0.1760
  recall@10    0.3360
  mrr@10       0.1277


This is where our theoretical framework stoppes working and reality kicks in. 

We will notice many more times that there are cases when one metric is not enough to tell the whole picture, even if we mapped it one to one in the methodology. If we think carefull about our NFR is to have **good ranking quality**. So far we have still do not know how good is *ndcg@10* - it became better but its still much to be desired. 


This means that there is something else we might have missed. This requires us to check other metrices. Always prefer standard metrices to which you know what it represents - we are flipping the script here and going bottom-up. Thus the most default metrices we can have at this level:
- recall@10
- mrr@10 

Notice that recall@10 is way higher than the ndcg@10 which means we can create a hypothesis: ranking is suboptimal, not the results. 

To check it more, lets get bigger sample of `metric@100`

In [14]:
from src.evaluation import evaluate_retrieval, RetrievalMetric

report_v2_100 = evaluate_retrieval(
    searcher_v2, 
    clerc_data,
    top_k=100,
    metrics=[
        RetrievalMetric.RECALL_100,
        RetrievalMetric.NDCG_100, 
        RetrievalMetric.MRR_100,
    ],
)  # smoke run
print(report_v2_100)

search:   0%|          | 0/2000 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Retrieval evaluation (2000 queries, top-100):
  recall@100   0.9765
  ndcg@100     0.3120
  mrr@100      0.1546
